# Line Model — Pretrained Code T5-small model with multiline context

In [1]:
%tb
import os, math, random, glob
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer, get_cosine_schedule_with_warmup
from torch.optim import AdamW

import warnings
warnings.filterwarnings("ignore")

from modules.plotting import MetricLog, plot_metrics
from modules.hand_testing import hand_test_repl
from modules.best_model_saver import BestModelSaver
from modules.early_stopping import EarlyStopping
from modules.datasets.loading import *
from modules.datasets.line_t5_multiline_dataset import *

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")

LINE_MODEL_NAME = "line_model_MLC_T5_small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

No traceback available to show.


WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Training loop

In [2]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(
    model:           T5ForConditionalGeneration,
    hf_tok:          AutoTokenizer,
    train_dl:        DataLoader,
    val_dl:          DataLoader,
    epochs:          int,
    lr:              float,
    device:          torch.device,
    saver:           BestModelSaver,
    log:             MetricLog,
    plot_dir:        str,
    label_smoothing: float = 0.1,
    warmup_frac:     float = 0.05,
    patience:        int   = 3,
    use_amp:         bool  = True,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    opt          = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    # T5 uses bfloat16 better than float16 — use bf16 if available, else fp16
    amp_enabled = use_amp and device.type == "cuda"
    amp_dtype   = torch.bfloat16 if (amp_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    scaler      = GradScaler("cuda", enabled=(amp_enabled and amp_dtype == torch.float16))
    tqdm.write(f"[AMP] enabled={amp_enabled}, dtype={amp_dtype}")

    stopper = EarlyStopping(patience=patience)

    for ep in range(1, epochs + 1):
        # ── TRAIN ────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_acc = t_steps = 0
        gn = 0.0

        batch_bar = tqdm(train_dl,
                         desc=f"[Line] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch")

        for src, mask, lbl in batch_bar:
            src, mask, lbl = src.to(device, non_blocking=True), mask.to(device, non_blocking=True), lbl.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                # T5 still does the label right-shift internally; we just override the loss
                out    = model(input_ids=src, attention_mask=mask, labels=lbl)
                logits = out.logits   # (B, T_dec, V) — aligned with lbl
                loss   = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    lbl.reshape(-1),
                    ignore_index=-100,
                    label_smoothing=label_smoothing,
                )

            if amp_dtype == torch.float16:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                gn = _clip_norm(model)
                scaler.step(opt)
                scaler.update()
            else:
                # bf16 doesn't need GradScaler
                loss.backward()
                gn = _clip_norm(model)
                opt.step()
            sched.step()

            with torch.no_grad():
                preds = logits.argmax(-1)
                valid = (lbl != -100)
                acc   = (preds[valid] == lbl[valid]).float().mean().item() if valid.any() else 0.0

            t_loss  += loss.item()
            t_acc   += acc
            t_steps += 1

            batch_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.3f}",
                                  lr=f"{opt.param_groups[0]['lr']:.2e}")

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        # ── VAL ──────────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for src, mask, lbl in tqdm(val_dl,
                                       desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                       leave=False, unit="batch"):
                src, mask, lbl = src.to(device, non_blocking=True), mask.to(device, non_blocking=True), lbl.to(device, non_blocking=True)
                with autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                    out    = model(input_ids=src, attention_mask=mask, labels=lbl)
                    logits = out.logits
                    loss   = F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        lbl.reshape(-1),
                        ignore_index=-100,
                        label_smoothing=label_smoothing,
                    )
                v_loss  += loss.item()
                v_steps += 1

        vl = v_loss / v_steps if v_steps else tl

        log.append(
            train_loss=tl, val_loss=vl,
            train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
            lr=opt.param_groups[0]["lr"],
            token_acc=ta, grad_norm=gn,
        )
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")

        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [3]:
class Arguments():
    def __init__(self, data_dir: str = f"{WORKDIR}/Clean_Dataset", ckpt_dir: str = f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                    plot_dir: str = F"{WORKDIR}/plots/{LINE_MODEL_NAME}", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    # args = Arguments(epochs=10)
    # args = Arguments(max_files=10)
    args = Arguments(max_files=100, epochs=2)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    HF_MODEL = "Salesforce/codet5-small"
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        lm = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME / f"{LINE_MODEL_NAME}_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(None, lm, None, hf_tok, device, supports_multiline=True)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]

    # ── LINE MODEL ───────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        line_model = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        
        collate    = lambda b: collate_t5(b, hf_tok.pad_token_id)
        tr_line_ds = T5LineDataset(tr_txt, hf_tok)
        va_line_ds = T5LineDataset(va_txt, hf_tok)
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.1f}M parameters (codet5-small)")
        
        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME, from_hf=True)
        line_log   = MetricLog()
        train_line_model(line_model, hf_tok, tr_line_dl, va_line_dl,
                         args.epochs, args.lr, device, line_saver, line_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(None, line_model, None, hf_tok, device, supports_multiline=True)


main()

[Loading] Started loading
[Data] loaded 100 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Prepairing LINE model
[T5LineDataset] 5941 samples (context_lines=3, max_prefix=384)
[T5LineDataset] 697 samples (context_lines=3, max_prefix=384)
[Line  Model] 60.5M parameters (codet5-small)
[Line] DataLoader — 186 train batches, 22 val batches
[AMP] enabled=True, dtype=torch.bfloat16


[Line  ep   1] train_loss=3.7821  val_loss=3.4695  ppl=32.1  acc=0.556  lr=2.70e-04


RuntimeError: [enforce fail at inline_container.cc:626] . unexpected pos 187472192 vs 187472080